%md
## 📑 HAPS PL/ NPL Waterfall

#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYA_HQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:

from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

# import delta_table_creator, extract, transform
# from delta import DeltaTable  # type: ignore

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

run_month_last_date = spark.sql(f'''select last_day({run_date!r}) as run_month_last_date''').collect()[0]['run_month_last_date'] 
run_month_last_date = run_month_last_date.strftime('%Y-%m-%d') 

PreviousMonthreferenceDate = spark.sql(f'''
                     select max(CalendarDate) as PreviousMonthreferenceDate
                     FROM `{env_prefix}-cdh`.reference.ref_calendar
                     WHERE LastDateOfMonth= last_day(add_months({run_date!r},-1)) and IsBusinessDay = 1
                     ''').collect()[0]['PreviousMonthreferenceDate']


# Create a DataFrame from your reference calendar table
calendar_df = spark.sql(f"SELECT * FROM `{env_prefix}-cdh`.reference.ref_calendar")

# Create a temporary view for the DataFrame
calendar_df.createOrReplaceTempView("calendar_df")                     

##### Schema of the table

In [0]:
table_name = 'Haps_PLNPLWaterfall'

schema = (
    "BusinessDate date, "
    "PortfolioName string, "
    "AgreementKey string, "
    "ExposureKey string, "
    "SOMBucket int, "
    "SOMGBVEUR decimal(19,2), "
    "MtDBucket int, "
    "MtDGBVEUR decimal(19,2), "
    "MtDPayment decimal(19,2), "
    "MtDLiquidations decimal(38,4), "
    "ModificationFlag int, "
    "GBVBeforeClosingDate double, "
    "GBVAtModificationDate decimal(19,2), "
    "Haircut decimal(19,2), "
    "Perimeter string, "
    "InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_month_last_date,  # type: ignore
)


##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_PLNPLWaterfall'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### portfolio

In [0]:
portfolio = spark.sql(
f'''
SELECT distinct P.BusinessDate,  p.PortfolioName, cast(P.AssignmentDate as date) AssignmentDate, p.unit_code_profits, p.prefix
FROM `{env_prefix}-cdh`.primary.portfolio P
INNER JOIN haps_portfolios  HP ON HP.PortfolioName = P.PortfolioName
WHERE P.BusinessDate = {run_date!r}
'''
)
portfolio.createOrReplaceTempView("portfolio")  

##### ActivePerimeterSoM

In [0]:
ActivePerimeterSoM  = spark.sql(
f'''
SELECT  
      A.BusinessDate
    , A.PortfolioName
    , A.AgreementKey
    , A.ExposureKey
    , A.ClosedDate
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) AS SoMGBVEUR
    , A.IsActiveFlag AS SoMIsActiveFlag
    , A.ProductKey 
    , A.DaysPastDueWithMateriality
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN portfolio p ON p.PortfolioName = A.PortfolioName
WHERE A.IsActiveFlag = true
AND A.BusinessDate = {PreviousMonthreferenceDate!r}
AND A.SourceSystem = 'profits'
''')
ActivePerimeterSoM.createOrReplaceTempView("ActivePerimeterSoM")

##### PerimeterMtD

In [0]:
PerimeterMtD  = spark.sql(
f'''
SELECT * 
    , RANK() OVER(PARTITION BY a.ExposureKey ORDER BY CASE WHEN A.MtDIsActiveFlag = true THEN 1 ELSE 0 END DESC) AS Rank
FROM (
SELECT  
      A.BusinessDate
    , A.PortfolioName
    , A.AgreementKey
    , A.ExposureKey
    , A.ClosedDate
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) AS MtDGBVEUR
    , A.IsActiveFlag AS MtDIsActiveFlag
    , A.ProductKey 
    , A.DaysPastDueWithMateriality
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN portfolio p ON p.PortfolioName = A.PortfolioName
WHERE A.IsActiveFlag = true
AND A.BusinessDate = {run_date!r}
AND A.SourceSystem = 'profits'

UNION ALL
    
SELECT  
      CAST({run_date!r} AS DATE) AS BusinessDate
    , A.PortfolioName
    , A.AgreementKey
    , A.ExposureKey
    , A.ClosedDate
    , NULL AS GBVEUR
    , A.IsActiveFlag AS MtDIsActiveFlag
    , A.ProductKey 
    , null as DaysPastDueWithMateriality
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN portfolio p ON p.PortfolioName = A.PortfolioName
WHERE A.IsActiveFlag = false
AND A.BusinessDate <= {run_date!r} AND A.BusinessDate > {PreviousMonthreferenceDate!r}
AND A.SourceSystem = 'profits') a
''')
PerimeterMtD.createOrReplaceTempView("PerimeterMtD")

In [0]:
from pyspark.sql.functions import when, col

def get_bucket_from_dpd(dpd):
    return (when(dpd.isNull(), None)
            .when(dpd <= 0, 0)
            .when((dpd >= 1) & (dpd <= 29), 1)
            .when((dpd >= 30) & (dpd <= 59), 2)
            .when((dpd >= 60) & (dpd <= 89), 3)
            .when((dpd >= 90) & (dpd <= 119), 4)
            .when((dpd >= 120) & (dpd <= 149), 5)
            .when((dpd >= 150) & (dpd <= 179), 6)
            .when((dpd >= 180) & (dpd <= 209), 7)
            .when((dpd >= 210) & (dpd <= 239), 8)
            .when((dpd >= 240) & (dpd <= 269), 9)
            .when((dpd >= 270) & (dpd <= 299), 10) 
            .when((dpd >= 300) & (dpd <= 329), 11)
            .when((dpd >= 330) & (dpd <= 359), 12)
            .when((dpd >= 360) & (dpd <= 720), 13)
            .when(dpd > 720, 14)
            .otherwise(None))

# Apply the transformation to PerimeterMtD DataFrame
PerimeterMtD = (
    PerimeterMtD
    .withColumn("MtDBucket", get_bucket_from_dpd(col("DaysPastDueWithMateriality")))
    .filter(col("Rank") == 1)
    .drop("Rank")
)
PerimeterMtD.createOrReplaceTempView("PerimeterMtD")

# Apply the transformation to ActivePerimeterSoM DataFrame
ActivePerimeterSoM = ActivePerimeterSoM.withColumn("SoMBucket", get_bucket_from_dpd(col("DaysPastDueWithMateriality")))
ActivePerimeterSoM.createOrReplaceTempView("ActivePerimeterSoM")

##### MonthlyPaymentsLiquidations

In [0]:
MonthlyPaymentsLiquidations  = spark.sql(
f'''
select ExposureKey
     , Sum(CASE WHEN PaymentTypeGroup not ilike '%liquidation%' THEN Amount ELSE 0 END) as MtDTotalAmountEUR
     , Sum(CASE WHEN PaymentTypeGroup ilike '%liquidation%' THEN Amount ELSE 0 END) as MtDTotalLiquidationAmountEUR
from (
    select distinct CollectionSourceKey
                  , ExposureKey 
                  , t.PaymentTypeGroup
                  , t.Amount
    from  `{env_prefix}-cdh`.derivative.dcollection t 
    where t.ActualDate <={run_date!r}
      and t.ActualDate > '{PreviousMonthreferenceDate}'
    )
group by ExposureKey
''')
MonthlyPaymentsLiquidations.createOrReplaceTempView("MonthlyPaymentsLiquidations")

##### GBV At Modification and Before Closing

In [0]:
GBV_per_businessdate = spark.sql(
    f'''
SELECT distinct A.ExposureKey
     , CASE WHEN A.Currency='EUR' THEN IFNULL(A.GBV,0) ELSE IFNULL(A.GBV,0)/B.ExchangeRate END GBVEUR
     , A.GBV
     , A.BusinessDate
     , A.Opendate
     , A.LastRestructuringDate
     , A.ClosedDate
FROM `{env_prefix}-cdh`.derivative.dexposure A
INNER JOIN portfolio p ON p.PortfolioName = A.PortfolioName
LEFT JOIN `{env_prefix}-cdh`.reference.exchangerates B ON A.BusinessDate=B.BusinessDate AND B.OriginalCurrency=A.Currency
WHERE A.BusinessDate <= {run_date!r} AND A.BusinessDate >= '{PreviousMonthreferenceDate}'
AND A.SourceSystem = 'profits'
''')
GBV_per_businessdate.createOrReplaceTempView("GBV_per_businessdate")

GBVAtModificationDate_0 = spark.sql(
    f'''
SELECT *
FROM calendar_df A
INNER JOIN GBV_per_businessdate B ON A.CalendarDate=CASE WHEN B.OpenDate > B.LastRestructuringDate THEN B.OpenDate ELSE B.LastRestructuringDate END
''')
GBVAtModificationDate_0.createOrReplaceTempView("GBVAtModificationDate_0")

GBVBeforeClosingDate_0 = spark.sql(
    f'''
SELECT DISTINCT 
      A.PreviousBusinessDay
    , A.CalendarDate
    , B.ClosedDate
    , B.BusinessDate
    , B.ExposureKey
FROM calendar_df A
INNER JOIN GBV_per_businessdate B ON A.CalendarDate=B.BusinessDate and B.ClosedDate IS NOT NULL
''')
GBVBeforeClosingDate_0.createOrReplaceTempView("GBVBeforeClosingDate_0")

GBVAtModificationDate = spark.sql(
    f'''
SELECT DISTINCT ExposureKey
     , GBV AS GBVAtModificationDate
     , GBVEUR AS GBVAtModificationDateEUR
     , BusinessDate
FROM GBVAtModificationDate_0 
wHERE BusinessDate=CASE WHEN IsBusinessDay = 0 THEN NextBusinessDay ELSE CalendarDate END 
'''
)
GBVAtModificationDate.createOrReplaceTempView("GBVAtModificationDate")

GBVBeforeClosingDate = spark.sql(
    f'''
SELECT DISTINCT A.ExposureKey
     , A.GBV AS GBVBeforeClosingDate
     , A.BusinessDate
     , A.GBVEUR AS GBVBeforeClosingDateEUR
FROM GBV_per_businessdate A
LEFT JOIN GBVBeforeClosingDate_0 B ON A.exposurekey=B.exposurekey
WHERE A.BusinessDate=B.PreviousBusinessDay
''')
GBVBeforeClosingDate.createOrReplaceTempView("GBVBeforeClosingDate")

##### ModificationsMapping

In [0]:
ModificationsMapping  = spark.sql(
f'''
select distinct 
    {run_date!r} as BusinessDate
    , res.PortfolioName
    , RES.OldAgreementKey
    , RES.NewAgreementKey
    , RES.OldExposureKey
    , RES.NewExposureKey
    , RES.RestructuringDate 
from `{env_prefix}-cdh`.primary.restructuring res
INNER JOIN portfolio p ON p.PortfolioName = res.PortfolioName
LEFT JOIN PerimeterMtD Ac on Ac.ExposureKey = res.NewExposureKey 
WHERE res.RestructuringDate <= {run_date!r}
and res.RestructuringDate >cast(last_day('{PreviousMonthreferenceDate}') as string)
and res.OldExposureKey is not null and OldExposureKey <> ''
AND RIGHT(case when AC.ProductKey is null then 'null' else AC.ProductKey end,4) NOT IN ('6078', '6079', '4015', '4016', '7362', '7363')
and LAST_DAY(res.BusinessDate) = {run_date!r}  
''')
ModificationsMapping.createOrReplaceTempView("ModificationsMapping")

##### Accounts

In [0]:
Accounts  = spark.sql(
f'''
select * 
from (
select 
  c.PortfolioName
, c.AgreementKey
, ifnull(a.OldExposureKey, c.ExposureKey) as OldExposureKey
, c.ExposureKey as NewExposureKey
, a.RestructuringDate
, CD.GBVBeforeClosingDateEUR
, MD.GBVAtModificationDateEUR
, b.SoMBucket 
, b.SoMGBVEUR
, b.SoMIsActiveFlag
, c.MtDGBVEUR
, c.MtDBucket
, c.MtDIsActiveFlag
, case when a.NewExposureKey is not null then 1 else 0 end as ModificationFlag
, IFNULL(ma1.MtDTotalAmountEUR,0) as MtDTotalAmountEURPrevious
, IFNULL(ma2.MtDTotalAmountEUR,0) as MtDTotalAmountEURNew
, IFNULL(MA1.MtDTotalLiquidationAmountEUR, 0) as MtDTotalLiquidationAmountEURPrevious
, IFNULL(MA2.MtDTotalLiquidationAmountEUR, 0) as MtDTotalLiquidationAmountEURNew
, (cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) END) * IFNULL(ma2.MtDTotalAmountEUR,0) AS MtDTotalAmountEURFinalNew
, (cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) END) * IFNULL(MA2.MtDTotalLiquidationAmountEUR, 0) AS MtDTotalLiquidationAmountEURFinalNew
--, cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) end as ratio
--, (cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) end) * c.MtDGBVEUR AS MtDGBVEURFinal
, case when (sum(b.SoMGBVEUR) OVER (PARTITION BY C.AgreementKey)) = 0 then ((cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY C.ExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY C.ExposureKey) end) * c.MtDGBVEUR)/ (count( b.ExposureKey) OVER (PARTITION BY C.AgreementKey)) else (cast(b.SoMGBVEUR as float)/ case when sum(b.SoMGBVEUR) OVER (PARTITION BY C.ExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY C.ExposureKey) end) * c.MtDGBVEUR end as MtDGBVEURFinal


--, (cast(b.SOMGBVEUR as float) / case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) end) * MD.GBVAtModificationDateEUR as GBVAtClosingDateFinal
, case when (cast(b.SOMGBVEUR as float) / case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) end) * MD.GBVAtModificationDateEUR is null and CD.GBVBeforeClosingDateEUR > 0 and MD.GBVAtModificationDateEUR > 0 /*and MM.NewExposureKey is not null*/ then MD.GBVAtModificationDateEUR else (cast(b.SOMGBVEUR as float) / case when sum(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) >0 THEN SUM(b.SoMGBVEUR) OVER (PARTITION BY A.NewExposureKey) end) * MD.GBVAtModificationDateEUR end as GBVAtClosingDateFinal
, case when a.NewExposureKey is not null then 'ModifiedAccounts'
       when b.ExposureKey = c.ExposureKey and a0.OldExposureKey is not null and b.SoMIsActiveFlag = true and c.MtDIsActiveFlag = true then 'ModifiedAccounts'
       when b.ExposureKey = c.ExposureKey and a0.OldExposureKey is null and b.SoMIsActiveFlag = true and c.MtDIsActiveFlag = true then 'CommonPerimeter'
       when b.ExposureKey = c.ExposureKey and a0.OldExposureKey is null and b.SoMIsActiveFlag = true and c.MtDIsActiveFlag = false then 'ClosedAccountWithinMonth' 
       when b.ExposureKey is null and c.MtDIsActiveFlag = true then 'NewAccountWithinMonth' 
  end as Perimeter
from PerimeterMtD c 
left join ModificationsMapping a on a.NewExposureKey = c.ExposureKey --AND RIGHT(c.ProductKey,4) NOT IN ('6078', '6079', '4015', '4016', '7362', '7363')
left join ModificationsMapping a0 on a0.OldExposureKey = c.ExposureKey
left join ActivePerimeterSoM b on ifnull(a.OldExposureKey, c.ExposureKey) = b.ExposureKey
left join MonthlyPaymentsLiquidations MA1 on b.ExposureKey = MA1.ExposureKey
left join MonthlyPaymentsLiquidations MA2 on c.ExposureKey = MA2.ExposureKey and b.ExposureKey is null
LEFT JOIN GBVAtModificationDate MD ON A.NewExposureKey = MD.ExposureKey
LEFT JOIN GBVBeforeClosingDate CD ON A.OldExposureKey = CD.ExposureKey) a 
WHERE a.Perimeter IS NOT NULL
''')
Accounts.createOrReplaceTempView("Accounts")

##### doublemod

In [0]:
doublemod  = spark.sql(
f'''
select a.AgreementKey, b.MtDGBVEUR/count(Distinct a.OldExposureKey) MtDGBVEUR
from Accounts a 
inner join (
  select AgreementKey, sum(case when MtDGBVEUR is null then  0 else MtDGBVEUR end) MtDGBVEUR
  from Accounts
  group by AgreementKey 
  having sum(case when MtDGBVEURFinal is null then  0 else MtDGBVEURFinal end ) = 0 
) b on a.AgreementKey = b.AgreementKey
where a.MtDIsActiveFlag = false
  and a.Perimeter = 'ModifiedAccounts' 
group by a.AgreementKey, b.MtDGBVEUR
having sum(case when MtDGBVEURFinal is null then  0 else MtDGBVEURFinal end ) = 0 
''')
doublemod.createOrReplaceTempView("doublemod")

##### Accounts_v2

In [0]:
Accounts_v2  = spark.sql(
f'''
select 
  a.PortfolioName
, a.AgreementKey
, a.OldExposureKey
, a.NewExposureKey
, a.RestructuringDate
, a.GBVBeforeClosingDateEUR
, a.GBVAtModificationDateEUR
, a.SoMBucket 
, a.SoMGBVEUR
, a.SoMIsActiveFlag
, a.MtDGBVEUR
, a.MtDBucket
, a.MtDIsActiveFlag
, a.ModificationFlag
, a.MtDTotalAmountEURPrevious
, a.MtDTotalAmountEURNew
, a.MtDTotalLiquidationAmountEURPrevious
, a.MtDTotalLiquidationAmountEURNew
,case when a.MtDTotalAmountEURFinalNew is null then b.MtDTotalAmountEURFinalNew2 else a.MtDTotalAmountEURFinalNew end as MtDTotalAmountEURFinalNew
,case when a.MtDTotalLiquidationAmountEURFinalNew is null then b.MtDTotalLiquidationAmountEURFinalNew2 else a.MtDTotalLiquidationAmountEURFinalNew end as MtDTotalLiquidationAmountEURFinalNew
--, a.ratio
, case when C.AgreementKey IS NOT NULL and a.MtDIsActiveFlag = false and a.GBVAtModificationDateEUR > 0 then C.MtDGBVEUR else a.MtDGBVEURFinal end as MtDGBVEURFinal
, a.GBVAtClosingDateFinal
, a.Perimeter
from Accounts a
left join (select * 
                , cast(MtDGBVEURFinal as float)/ case when sum(MtDGBVEURFinal) OVER (PARTITION BY NewExposureKey) >0 
		                                  THEN SUM(MtDGBVEURFinal) OVER (PARTITION BY NewExposureKey) end * MtDTotalAmountEURNew as MtDTotalAmountEURFinalNew2
		            , cast(MtDGBVEURFinal as float)/ case when sum(MtDGBVEURFinal) OVER (PARTITION BY NewExposureKey) >0 
		                                  THEN SUM(MtDGBVEURFinal) OVER (PARTITION BY NewExposureKey) end * MtDTotalLiquidationAmountEURNew as MtDTotalLiquidationAmountEURFinalNew2
		       from Accounts
		       where MtDTotalAmountEURFinalNew is null 
 ) b on a.NewExposureKey = b.NewExposureKey and a.OldExposureKey = b.OldExposureKey
left join doublemod C on a.AgreementKey = C.AgreementKey
''')
Accounts_v2.createOrReplaceTempView("Accounts_v2")

In [0]:
%sql
select * from Accounts_v2 where NewExposureKey = 'GLB989003385977606' or OldExposurekey = 'GLB989003385977606'

##### TEMP

In [0]:
TEMP = spark.sql(
f'''
select distinct 
       last_day({run_date!r}) as BusinessDate,
       PortfolioName, 
       AgreementKey, 
       case when Perimeter in('ClosedAccountWithinMonth','ModifiedAccounts') then OldExposureKey else NewExposureKey end as Exposurekey, 
       case when SOMBucket is null then 0 else SOMBucket end as SOMBucket, 
       case when SOMGBVEUR is null then 0 else SOMGBVEUR end as SOMGBVEUR, 
       MtDBucket, 
       case when Perimeter = 'ModifiedAccounts' then ifnull(MtDGBVEURFinalTotal,0) else ifnull(MtDGBVEUR,0) end as MtDGBVEUR,
       case when Perimeter = 'ClosedAccountWithinMonth'  then ifnull(MtDTotalAmountEURPrevious,0)
            when Perimeter = 'ModifiedAccounts' then ifnull(MtDTotalAmountEURPrevious,0) + ifnull(MtDTotalAmountEURFinalNew,0)
            else ifnull(MtDTotalAmountEURPrevious,0)
       end as MtDPayment,
       --ifnull(MtDTotalAmountEURPrevious,0) + ifnull(MtDTotalAmountEURFinalNew,0) as MtDPayment,
       ifnull(MtDTotalLiquidationAmountEURPrevious,0) as MtDLiquidations,       
       case when Perimeter = 'ModifiedAccounts' then ModificationFlag else 0 end as ModificationFlag,
       case when Perimeter = 'ModifiedAccounts' then ifnull(GBVBeforeClosingDate,0) else 0 end as GBVBeforeClosingDate,
       case when Perimeter = 'ModifiedAccounts' then ifnull(GBVAtModificationDate,0) else 0 end as GBVAtModificationDate,
       case when Perimeter = 'ModifiedAccounts' then ifnull(GBVBeforeClosingDate,0) - ifnull(GBVAtModificationDate,0) else 0 end as Haircut,
       cast(0.00 as decimal(19,2)) as AssetSales,
       Perimeter  
from (
        select * , 
               GBVBeforeClosingDateEUR as  GBVBeforeClosingDate, 
               sum(GBVAtClosingDateFinal) over(partition by OldExposureKey) as GBVAtModificationDate,
               sum(MtDGBVEURFinal) over(partition by OldExposureKey) as MtDGBVEURFinalTotal

              --  , 
              --  sum(MtDTotalAmountEURFinalNew) over(partition by OldExposureKey) as MtDTotalAmountEURNew

               
from Accounts_v2
) m
where Perimeter in('CommonPerimeter','ClosedAccountWithinMonth','NewAccountWithinMonth') or (Perimeter = 'ModifiedAccounts' and ModificationFlag = 1)
''')
TEMP.createOrReplaceTempView("TEMP")

##### PLNPLWaterfall

In [0]:
FLDT_restructures = spark.sql(
f'''
SELECT DISTINCT CAST(OldExposureKey AS STRING) AS OldExposureKey
FROM  `{env_prefix}-cdh`.primary.restructuring 
WHERE BusinessDate = {run_date!r}
AND LAST_DAY(RestructuringDate) = LAST_DAY({run_date!r})
'''
)
FLDT_restructures.createOrReplaceTempView("FLDT_restructures")

fldt_allapplications = spark.sql(
f'''
SELECT DISTINCT * FROM (
SELECT distinct  unifiedLoan_AccountNumberWithPrefix as OldExposureKey
               , product_HairCutAmount
               , rank()over(partition by unifiedLoan_AccountNumberWithPrefix order by QualcoAppDisbursedDate DESC) rank
FROM `{env_prefix}-cdh-delivery`.repoint.haps_fldtallapplications AS app
WHERE APP.ActivatedApplicationFlag = 1
AND  app.BusinessDate = {run_date!r}
AND unifiedLoan_AccountNumberWithPrefix is not null
AND LAST_DAY(app.profitsInfo_DateCreated) = LAST_DAY({run_date!r}) 
AND CONCAT(LEFT(app.unifiedLoan_AccountNumberWithPrefix,3),app.profitsInfo_NewAccountNumber)  IS NOT NULL
AND lower(app.product_ProductCategoryType) NOT LIKE '_dpo'
)A WHERE rank=1
'''
)
fldt_allapplications.createOrReplaceTempView("fldt_allapplications")


In [0]:
PLNPLWaterfall = spark.sql(
f'''
select BusinessDate 
    , PortfolioName
    , AgreementKey
    , ExposureKey
    , SOMBucket
    , SOMGBVEUR
    , case when MtDBucket is null then 0 else MtDBucket end as MtDBucket
    , MtDGBVEUR
    , MtDPayment
    , MtDLiquidations
    , ModificationFlag
    , GBVBeforeClosingDate
    , GBVAtModificationDate
    , case when fldt.product_HairCutAmount is null then 0 else fldt.product_HairCutAmount end as Haircut
    , Perimeter
    , AssetSales 
from TEMP a
left join FLDT_restructures fldt_res on a.ExposureKey = fldt_res.OldExposureKey 
left join fldt_allapplications fldt on fldt_res.OldExposureKey = fldt.OldExposureKey
''')
PLNPLWaterfall.createOrReplaceTempView("PLNPLWaterfall")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
    CAST(BusinessDate              AS DATE)           AS BusinessDate,
    CAST(PortfolioName             AS STRING)         AS PortfolioName,
    CAST(AgreementKey              AS STRING)         AS AgreementKey,
    CAST(ExposureKey               AS STRING)         AS ExposureKey,
    CAST(SOMBucket                 AS INT)            AS SOMBucket,
    CAST(SOMGBVEUR                 AS DECIMAL(19,2))  AS SOMGBVEUR,
    CAST(MtDBucket                 AS INT)            AS MtDBucket,
    CAST(MtDGBVEUR                 AS DECIMAL(19,2))  AS MtDGBVEUR,
    CAST(MtDPayment                AS DECIMAL(19,2))  AS MtDPayment,
    CAST(MtDLiquidations           AS DECIMAL(38,4))  AS MtDLiquidations,
    CAST(ModificationFlag          AS INT)            AS ModificationFlag,
    CAST(GBVBeforeClosingDate      AS DOUBLE)         AS GBVBeforeClosingDate,
    CAST(GBVAtModificationDate     AS DECIMAL(19,2))  AS GBVAtModificationDate,
    CAST(Haircut                   AS DECIMAL(19,2))  AS Haircut,
    CAST(Perimeter                 AS STRING)         AS Perimeter,
    CAST(getdate()                 AS TIMESTAMP)      AS InsertTimeStamp
FROM PLNPLWaterfall 
''')
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_month_last_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_month_last_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_month_last_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_month_last_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_month_last_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PLNPLWaterfall = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_month_last_date!r}
'''
)
PLNPLWaterfall.createOrReplaceTempView("PLNPLWaterfall")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/PLNPLWaterfall/" # 
file_name = "PLNPLWaterfall_" +run_month_last_date+ ".csv"
PLNPLWaterfall.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)